# AIA 193 density jump and Alfvén Mach number (intensity-ratio method)

Optically-thin EUV emission is $I=\int n_e^2 G(T)\,\mathrm{d}\ell$. For a
quasi-isothermal compression of roughly constant temperature and line-of-sight
depth, $I\propto n_e^2$, so the density compression in a box ahead of the wave is

$$X\equiv n_2/n_1=\sqrt{I_2/I_1}.$$

The perpendicular fast-mode Rankine–Hugoniot relation in the low-$\beta$ limit
($\gamma=5/3$) then gives the Alfvén Mach number — the same `alfven_mach_from_X`
used for the type II band-splitting:

$$M_A=\sqrt{\dfrac{X(X+5)}{2(4-X)}},\qquad 1\le X<4.$$

AIA L1.5 is loaded locally with `load_channel` (no Fido). ROI bounds are variables
cropped with `crop_to_roi`.

**Caveats.** $I\propto n_e^2$ ignores the temperature dependence of $G(T)$, the LOS
integration through unshocked plasma, and depth changes; cross-check strong
compressions against the DEM pipeline.

## 0. Imports and parameters

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import logging
import sunpy
sunpy.log.setLevel(logging.WARNING)

import os, glob
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord
import sunpy.map
from sunpy.map import Map
from sunpy.time import parse_time
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

data_dir = '/home/mnedal/data'
date     = '2025-10-06'


### Shared helpers (identical to `plot_jmap`)

In [ ]:
def load_channel(channel):
    """Load local AIA L1.5 files for one channel as a MapSequence (same as plot_jmap)."""
    y, m, d = date.split('-')
    files = sorted(glob.glob(
        f'{data_dir}/AIA/{channel}A/highres/lv15/'
        f'aia.lev15.{channel}A_{y}_{m}_{d}T*_lev15*.fits'))
    if not files:
        raise FileNotFoundError(f'No files for {channel} A on {date}.')
    return sunpy.map.Map(files, sequence=True)


def crop_to_roi(rmap, box):
    """Submap to an ROI given as dict(left, right, bottom, top) in arcsec (same as plot_jmap)."""
    frame = rmap.coordinate_frame
    bl = SkyCoord(box['left']  * u.arcsec, box['bottom'] * u.arcsec, frame=frame)
    tr = SkyCoord(box['right'] * u.arcsec, box['top']    * u.arcsec, frame=frame)
    return rmap.submap(bl, top_right=tr)


## 1. Box-mean intensity, density jump and Mach number

In [ ]:
def alfven_mach_from_X(X, gamma=5.0/3.0):
    """Perpendicular-shock Alfven Mach number from density jump X (gamma=5/3).

    M_A^2 = X (X + 5) / (2 (4 - X)), valid for 1 <= X < 4. Same relation as the
    type II band-splitting analysis.
    """
    X = np.asarray(X, dtype=float)
    out = np.full_like(X, np.nan)
    ok = (X >= 1) & (X < 4)
    out[ok] = np.sqrt(X[ok]*(X[ok] + 5) / (2*(4 - X[ok])))
    return out


def box_mean_timeseries(seq, box):
    """Mean intensity inside an ROI dict box across a MapSequence."""
    times, vals = [], []
    for m in seq.maps:
        sub = crop_to_roi(m, box)
        vals.append(np.nanmean(sub.data)); times.append(m.date.datetime)
    return pd.to_datetime(times), np.asarray(vals)


def density_jump_series(times, box_int, baseline_end):
    """X(t) = sqrt(I(t)/I_pre); I_pre = mean box intensity before baseline_end."""
    pre = box_int[times < pd.to_datetime(baseline_end)]
    I1 = np.nanmean(pre)
    return np.sqrt(box_int / I1), I1


In [ ]:
aia193_seq = load_channel(193)        # local L1.5 MapSequence

# Small box just ahead of the wave front (AIA arcsec) as ROI-dict variables.
BOX_ROI      = dict(left=180, right=240, bottom=120, top=180)
baseline_end = f'{date}T08:48:00'     # pre-event baseline ends here

t_box, I_box = box_mean_timeseries(aia193_seq, BOX_ROI)
X_t, I1 = density_jump_series(t_box, I_box, baseline_end=baseline_end)
MA_t = alfven_mach_from_X(X_t)


In [ ]:
# Show the box on one frame.
i_mid = len(aia193_seq.maps)//2
m = aia193_seq.maps[i_mid]
bl = SkyCoord(BOX_ROI['left']*u.arcsec, BOX_ROI['bottom']*u.arcsec, frame=m.coordinate_frame)
tr = SkyCoord(BOX_ROI['right']*u.arcsec, BOX_ROI['top']*u.arcsec, frame=m.coordinate_frame)
fig = plt.figure(figsize=[7, 7]); ax = fig.add_subplot(projection=m)
m.plot(axes=ax, clip_interval=(1, 99.9)*u.percent)
m.draw_quadrangle(bl, top_right=tr, axes=ax, edgecolor='cyan')
ax.grid(False); plt.show()


In [ ]:
fig, axs = plt.subplots(2, 1, figsize=[8, 7], sharex=True)
axs[0].plot(t_box, X_t, 'o-', ms=3, color='tab:blue')
axs[0].axhline(1, color='k', lw=0.8, alpha=0.5)
axs[0].set_ylabel('Density jump $X=n_2/n_1$')
axs[1].plot(t_box, MA_t, 'o-', ms=3, color='tab:red')
axs[1].axhline(1, color='k', lw=0.8, alpha=0.5)
axs[1].set_ylabel('Alfvén Mach number $M_A$')
axs[1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
axs[1].set_xlabel('Time [UT]')
fig.autofmt_xdate(); fig.suptitle('AIA 193 intensity-ratio shock diagnostics')
fig.tight_layout(); plt.show()

pd.DataFrame({'time': t_box, 'box_intensity': I_box, 'X': X_t, 'M_A': MA_t}).to_csv(
    './aia193_density_jump_mach.csv', index=False)
